# CQAS Anomaly Detection Analysis

This notebook demonstrates the CQAS anomaly detection workflow using synthetic data.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.anomaly.baseline_model import BaselineModel
from src.anomaly.delta_checker import DeltaChecker

## 1. Generate Synthetic Baseline Data

In [ ]:
rng = np.random.default_rng(42)
# Simulate 30 days of hourly failed login counts
normal_data = rng.normal(loc=5, scale=2, size=720).clip(0)  # 720 hours = 30 days
# Inject anomalies: spike on days 25-26
anomaly_data = normal_data.copy()
anomaly_data[600:612] = rng.normal(loc=35, scale=5, size=12)

print(f'Normal data: mean={normal_data.mean():.2f}, std={normal_data.std():.2f}')
print(f'With anomalies: range=[{anomaly_data.min():.1f}, {anomaly_data.max():.1f}]')

## 2. Build Baseline Model

In [ ]:
model = BaselineModel(baseline_dir='../data/baselines', min_data_points=50)
model.fit('failed_logins', {'10.0.0.1': normal_data.tolist()})
bl = model.get_baseline('failed_logins', '10.0.0.1')
print(f'Baseline: mean={bl["mean"]:.2f}, std={bl["std"]:.2f}')

## 3. Detect Anomalies with Z-Score Thresholding

In [ ]:
z_scores = [(v - bl['mean']) / bl['std'] for v in anomaly_data]
anomaly_mask = [abs(z) > 2.5 for z in z_scores]
print(f'Anomalies detected: {sum(anomaly_mask)} out of {len(anomaly_data)} observations')

## 4. Visualize Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

hours = range(len(anomaly_data))
ax1.plot(hours, anomaly_data, label='Failed Logins', color='steelblue', alpha=0.7)
ax1.axhline(bl['mean'], color='green', linestyle='--', label=f'Baseline Mean ({bl["mean"]:.1f})')
ax1.axhline(bl['mean'] + 2.5*bl['std'], color='orange', linestyle='--', label='Warning (z=2.5)')
ax1.axhline(bl['mean'] + 3.5*bl['std'], color='red', linestyle='--', label='Critical (z=3.5)')
anomaly_hours = [h for h, a in zip(hours, anomaly_mask) if a]
anomaly_values = [anomaly_data[h] for h in anomaly_hours]
ax1.scatter(anomaly_hours, anomaly_values, color='red', zorder=5, label=f'Anomalies ({len(anomaly_hours)})', s=50)
ax1.set_xlabel('Hour')
ax1.set_ylabel('Failed Login Count')
ax1.set_title('Failed Login Anomaly Detection (10.0.0.1)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(hours, z_scores, color='purple', alpha=0.7, label='Z-Score')
ax2.axhline(2.5, color='orange', linestyle='--', label='Warning threshold')
ax2.axhline(-2.5, color='orange', linestyle='--')
ax2.axhline(3.5, color='red', linestyle='--', label='Critical threshold')
ax2.axhline(-3.5, color='red', linestyle='--')
ax2.fill_between(hours, -2.5, 2.5, alpha=0.1, color='green', label='Normal range')
ax2.set_xlabel('Hour')
ax2.set_ylabel('Z-Score')
ax2.set_title('Z-Score Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed_logs/anomaly_detection_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')